In [20]:
from collections import defaultdict
from typing import Hashable, Iterable, Sequence, Tuple, Dict, List, Literal

In [21]:
Token = Hashable
Ngram = Tuple[Token, ...]

In [22]:
def _group_ngrams_by_len(common_max_ngrams: Iterable[Sequence[Token]]) -> Dict[int, set[Ngram]]:
    by_len: Dict[int, set[Ngram]] = defaultdict(set)
    for ng in common_max_ngrams:
        t = tuple(ng)
        if len(t) > 0:
            by_len[len(t)].add(t)
    return dict(by_len)


def _merge_intervals(intervals: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    if not intervals:
        return []
    intervals.sort()
    merged = [intervals[0]]
    for s, e in intervals[1:]:
        ps, pe = merged[-1]
        if s <= pe:
            merged[-1] = (ps, max(pe, e))
        else:
            merged.append((s, e))
    return merged


def _covered_token_count(tokens: Sequence[Token], ngrams_by_len: Dict[int, set[Ngram]]) -> int:
    L = len(tokens)
    if L == 0 or not ngrams_by_len:
        return 0

    intervals: List[Tuple[int, int]] = []

    # For each length, do one sliding pass and record matched spans.
    for n, grams in ngrams_by_len.items():
        if n > L:
            continue
        # Slide windows of size n
        for i in range(0, L - n + 1):
            if tuple(tokens[i : i + n]) in grams:
                intervals.append((i, i + n))

    # Union spans (prevents double-counting overlapping matches)
    merged = _merge_intervals(intervals)
    return sum(e - s for s, e in merged)


def weighted_max_common_score(
    q_tokens: Sequence[Token],
    k_tokens: Sequence[Token],
    common_max_ngrams: Iterable[Sequence[Token]],
    *,
    overall: Literal["harmonic", "mean", "min", "max"] = "harmonic",
    decimals: int = 6,
) -> Dict[str, float]:
    """
    Best-fit score for your setup:
      - uses ONLY maximal common n-grams you already computed
      - weights by n automatically via token coverage
      - symmetric overall score (recommended: harmonic mean)

    Returns: {score, q_cov, k_cov}
    """
    ngrams_by_len = _group_ngrams_by_len(common_max_ngrams)

    if not q_tokens or not k_tokens or not ngrams_by_len:
        return {"score": 0.0, "q_cov": 0.0, "k_cov": 0.0}

    q_cov = _covered_token_count(q_tokens, ngrams_by_len) / len(q_tokens)
    k_cov = _covered_token_count(k_tokens, ngrams_by_len) / len(k_tokens)

    if overall == "harmonic":
        score = 0.0 if (q_cov + k_cov) == 0 else (2 * q_cov * k_cov) / (q_cov + k_cov)
    elif overall == "mean":
        score = (q_cov + k_cov) / 2
    elif overall == "min":
        score = min(q_cov, k_cov)
    elif overall == "max":
        score = max(q_cov, k_cov)
    else:
        raise ValueError("overall must be 'harmonic', 'mean', 'min', or 'max'.")

    return {
        "score": round(score, decimals),
        "q_cov": round(q_cov, decimals),
        "k_cov": round(k_cov, decimals),
    }

In [23]:
import sys

import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from model_loading import load_model
from read_and_write_docs import read_txt
from n_gram_tracing import (
    common_ngrams
)

In [24]:
tokenizer = load_model("/Volumes/BCross/models/gpt2", load_model=False)

In [25]:
known_text = read_txt("../../data/kevin_hyatt_mail_1.txt")
unknown_text = read_txt("../../data/kevin_hyatt_mail_2.txt")

In [26]:
common = common_ngrams(
    text1=known_text,
    text2=unknown_text,
    n=1,
    tokenizer=tokenizer,
    include_subgrams=False,
    lowercase=True
)

Token indices sequence length is longer than the specified maximum sequence length for this model (1098 > 1024). Running this sequence through the model will result in indexing errors


In [27]:
def get_tokens(
    text: str,
    tokenizer,
    *,
    lowercase: bool = True,
    add_special_tokens: bool = False,
):
    """
    Hugging Face Transformers tokenizer -> list of token strings.

    Uses:
      - tokenizer.encode(..., add_special_tokens=...)  :contentReference[oaicite:0]{index=0}
      - tokenizer.convert_ids_to_tokens(...)            :contentReference[oaicite:1]{index=1}
    """
    if lowercase:
        text = text.lower()

    input_ids = tokenizer.encode(text, add_special_tokens=add_special_tokens)
    return tokenizer.convert_ids_to_tokens(input_ids)

In [28]:
unknown_tokens = get_tokens(unknown_text, tokenizer)
known_tokens = get_tokens(known_text, tokenizer)

In [29]:
weighted_max_common_score(unknown_tokens, known_tokens, common)

{'score': 0.370118, 'q_cov': 0.337887, 'k_cov': 0.409147}